In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack


from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import cohen_kappa_score
from scipy.stats import pearsonr

In [2]:
df = pd.read_csv("task1_cleaned_final.csv")

df = df.dropna(subset=[
    "Essay",
    "Task_Achievement",
    "Coherence_Cohesion",
    "Lexical_Resource",
    "Range_Accuracy"
]).reset_index(drop=True)

features = [
    "word_count",
    "unique_words",
    "ttr",
    "avg_word_len",
    "sentence_count",
    "avg_sentence_len",
    "sentence_len_var",
    "long_word_ratio",
    "short_word_ratio",
    "punct_density",
]

print("Total Len:", len(df))

Total Len: 8327


In [3]:
X_text = df["Essay"].values

y = df[[
    "Task_Achievement",
    "Coherence_Cohesion",
    "Lexical_Resource",
    "Range_Accuracy"
]].values

feat_values = df[features].values

In [4]:
idx = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    idx,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=2/3,
    random_state=42
)

X_train = X_text[train_idx]
X_val   = X_text[val_idx]
X_test  = X_text[test_idx]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

feat_train = feat_values[train_idx]
feat_val   = feat_values[val_idx]
feat_test  = feat_values[test_idx]

print(len(train_idx), len(val_idx), len(test_idx))


5828 833 1666


In [5]:
scaler_feat = StandardScaler()

feat_train = scaler_feat.fit_transform(feat_train)
feat_val   = scaler_feat.transform(feat_val)
feat_test  = scaler_feat.transform(feat_test)

In [6]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,1),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)


X_train_full = hstack([X_train_vec, feat_train])
X_val_full   = hstack([X_val_vec, feat_val])
X_test_full  = hstack([X_test_vec, feat_test])

In [7]:
model = MultiOutputRegressor(LinearRegression())

In [8]:
model.fit(X_train_full, y_train)


,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,LinearRegression()
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [9]:
y_val_pred = model.predict(X_val_full)
y_val_pred = np.clip(y_val_pred, 1, 9)

val_mae = mean_absolute_error(y_val, y_val_pred)

print("\nVALIDATION RESULT")
print("MAE:", val_mae)


VALIDATION RESULT
MAE: 1.9424103493108251


In [10]:
y_pred = model.predict(X_test_full)
y_pred = np.clip(y_pred, 1, 9)

In [11]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

pearson = pearsonr(y_test.flatten(), y_pred.flatten())[0]

rounded_preds = np.round(y_pred * 2) / 2

within_half = np.mean(np.abs(rounded_preds - y_test) <= 0.5)

true_band = np.round(y_test.mean(axis=1) * 2).astype(int)
pred_band = np.round(y_pred.mean(axis=1) * 2).astype(int)

qwk = cohen_kappa_score(true_band, pred_band, weights="quadratic")

print("\nTEST RESULTS")

print("MAE:", mae)
print("RMSE:", rmse)
print("Pearson:", pearson)
print("Within ±0.5:", within_half)
print("QWK:", qwk)


TEST RESULTS
MAE: 1.9591920891566132
RMSE: 2.5216266822467612
Pearson: 0.29120027809254306
Within ±0.5: 0.2537515006002401
QWK: 0.28293991454650924


In [12]:
def predict_task1_linear(essay):

    tokens = essay.split()
    
    word_count = len(tokens)
    unique_words = len(set(tokens))
    ttr = unique_words / word_count if word_count > 0 else 0
    avg_word_len = np.mean([len(w) for w in tokens]) if word_count > 0 else 0
    
    sentences = essay.split(".")
    sentence_count = len([s for s in sentences if s.strip() != ""])
    avg_sentence_len = word_count / max(sentence_count,1)
    
    sentence_len_var = np.var([len(s.split()) for s in sentences if s.strip() != ""]) if sentence_count > 0 else 0
    
    long_word_ratio = np.mean([len(w)>6 for w in tokens]) if word_count > 0 else 0
    short_word_ratio = np.mean([len(w)<=3 for w in tokens]) if word_count > 0 else 0
    
    punct_density = len([c for c in essay if c in ",.!?;:"]) / max(word_count,1)
    
    feat = [
        word_count,
        unique_words,
        ttr,
        avg_word_len,
        sentence_count,
        avg_sentence_len,
        sentence_len_var,
        long_word_ratio,
        short_word_ratio,
        punct_density
    ]
    
    feat = scaler_feat.transform([feat])
    
    vec = vectorizer.transform([essay])
        
    vec_full = hstack([vec, feat])
    
    pred = model.predict(vec_full)[0]

    pred = np.clip(pred, 1, 9)

    pred = [round(s * 2) / 2 for s in pred]

    overall = round(np.mean(pred) * 2) / 2

    return {
        "Task_Response": pred[0],
        "Coherence_Cohesion": pred[1],
        "Lexical_Resource": pred[2],
        "Grammatical_Range_Accuracy": pred[3],
        "Overall": overall
    }

In [13]:
band_7 = """ The line charts indicate how the number of train passengers varied in an unspecified geographical location, as well as revealing the proportion of trains running punctually compared to the fixed target of 95%. All the statistics are recorded for the period from 2000 to 2009.

From a quick glance, the number of train passengers underwent considerable fluctuation, reaching a peak in 2005. Meanwhile, the rate of trains running on time shared the same statistical progression, meeting or exceeding the target from 2002 to 2005 and again from 2008 to 2009.

In 2000, the number of travelers using trains was 37 million, climbing to around 42 million before dropping to just below its starting number in 2003. During the next years, train passengers rose sharply and hit the most significant peak of around 47 million in 2005; afterwards, the figure was somewhat unsteady, ending up staying at around 43 million at the end of the period.

In terms of running time efficiency, the rate stood at 92% within the year 2000, subsequently amounting to an adequate 95% in 2002. In the later years, the proportion of on-time trains exceeded the set target when the rate was 96% in 2004, but later declined notably by 4% in 2006. However, the percentage of punctual trains went up gradually and eventually stabilized at 97%."""


print("Band 7 Prediction:", predict_task1_linear(band_7))

Band 7 Prediction: {'Task_Response': 9.0, 'Coherence_Cohesion': 9.0, 'Lexical_Resource': 9.0, 'Grammatical_Range_Accuracy': 2.5, 'Overall': 7.5}
